# User Review

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/User_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
# @title Load Git Repository
!git clone https://github.com/PeaceAndLongLife/Analysis-Colab.git
%cd Analysis-Colab

import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))
%cd notebooks

### Setup Google API

In [ ]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
sys.path.append('src')

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')

# 3. Import your file!
from GoogleFunctions import GoogleDocumentManager, extract_file_id
from local_io import read_csv_from_id


### Read in data

data read in will be merged using the `user` column

- consent file
- transcript data

In [ ]:
# UserProfile Info
# @markdown Read in userprofile file
userprofile_file_link = "https://drive.google.com/file/d/1TGJ-6OP2ig5eQeT8722i6VOn2-g250di/view?usp=drive_link" # @param {"type":"string"}
userprofile_file_link_id = extract_file_id(userprofile_file_link)
show_userprofile = False # @param {"type":"boolean"}

# Consent data info
# @markdown Read in consent file

consent_file_link = "https://drive.google.com/file/d/1OBOz23ByNoxk7L2-DMKa3rhR_U9G56b-/view?usp=drive_link" # @param {"type":"string"}
consent_file_link_id = extract_file_id(consent_file_link)
show_consent = False # @param {"type":"boolean"}

# Transcript Data info
# @markdown Read in transcript file
trans_file_link = "https://drive.google.com/file/d/1Q4G-RJAXd0GgkCyJpA8ajTXTk0hOyQhe/view?usp=drive_link" # @param {"type":"string"}
trans_file_link_id = extract_file_id(trans_file_link)
show_trans = False # @param {"type":"boolean"}

# Lab File info
# @title Read in labfile file
labfile_file_link = "https://drive.google.com/file/d/1_lj9joFbkBjqZOXT7z7-q2AkFbc6sJez/view?usp=drive_link" # @param {"type":"string"}
labfile_file_link_id = extract_file_id(labfile_file_link)
show_labfile = False # @param {"type":"boolean"}

# Grades File
# @title Read in Grades file {"form-width":"20%"}
grades_file_link = "https://drive.google.com/file/d/1sOAi7UKfeYjv_l6DjsHfMbEQJbK4p9Hz/view?usp=drive_link" # @param {"type":"string"}
grades_file_link_id = extract_file_id(grades_file_link)
show_grades = False # @param {"type":"boolean"}

userprofile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, userprofile_file_link_id, show_userprofile)
consent_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, consent_file_link_id, show_consent)
trans_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans_file_link_id, show_trans)
labfile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, labfile_file_link_id, show_labfile)
grades_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, grades_file_link_id, show_grades)

from data_scrub import process_messages, parse_messages, explode_json_messages
import pandas as pd
import re

###
# Merge consent into transcript
combined_df = process_messages(
    consent_df,
    trans_df,
    pattern = r"(.+) - Assignment (\d+), Question: (\d+) - (.+)",
)
# Apply the parsing function to the 'all_messages' column
combined_df['all_messages'] = combined_df['all_messages'].apply(parse_messages)

# Count message objects in 'all_messages' and add to 'interactions' column
combined_df['interactions'] = combined_df['all_messages'].apply(len)

combined_df = combined_df[combined_df['all_messages'].apply(lambda x: len(x) > 0)]

print("Combined DataFrame with split question columns and interactions:")
# display(combined_df.head())

combined_df = explode_json_messages(combined_df)

###
# Merge consent into labfile_df manually to avoid the 'question_number' KeyError in process_messages
merged_lab_df = pd.merge(consent_df, labfile_df, on='user', how='inner')

# Define the pattern for lab files
lab_pattern = r"(.+) - Assignment (\d+)"

# Extract Course and lab_number using the pattern from the 'Assignment' column
extracted_lab_data = merged_lab_df['Assignment'].str.extract(lab_pattern)
merged_lab_df[['Course', 'lab_number']] = extracted_lab_data

# Convert lab_number to numeric
merged_lab_df['lab_number'] = pd.to_numeric(merged_lab_df['lab_number'], errors='coerce')
merged_lab_df['lab_number'] = merged_lab_df['lab_number'].astype(int).astype(str).str.zfill(2)

# Assign to combined_lab_df
combined_lab_df = merged_lab_df


### Process data

### Filter functions

In [ ]:
# @title Dropdown Filters {"form-width":"20%"}
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# --- Used Data Dropdown ---
used_data = widgets.ToggleButton(
    value=False,
    description='Used Data',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Select to filter results on if Ai used student data.',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)
if used_data.value:
  data_filtered_df = combined_df[combined_df['data'] == used_data.value]
else:
  data_filtered_df = combined_df

# --- User Dropdown ---
# Get unique users values
unique_users_all = ['Select User'] + sorted(data_filtered_df['user'].dropna().unique().tolist())
user_dropdown = widgets.Dropdown(
    options=unique_users_all,
    value='Select User', # Initial value
    description='Select User:',
    disabled=False,
)

# --- Course Dropdown ---
# Initialize with default options; on_user_change will populate it
course_dropdown = widgets.Dropdown(
    options=['Select Course'], # Initial options,
    value='Select Course', # Initial value
    description='Select Course:',
    disabled=True,
)

# --- Lab Dropdown ---
# Initialize with default options; on_course_change will populate it
lab_dropdown = widgets.Dropdown(
    options=['Select Lab'], # Initial options
    value='Select Lab', # Initial value
    description='Select Lab:',
    disabled=True,
)

# --- Question Dropdown ---
# Initialize with default options; on_lab_change will populate it
question_dropdown = widgets.Dropdown(
    options=['Select Question'], # Initial options
    value='Select Question', # Initial value
    description='Select Question:',
    disabled=True,
)

doc_button = widgets.Button(
    description='Report',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Open Lab Report',
    icon='file-text-o' # (FontAwesome names without the `fa-` prefix)
)

sheet_button = widgets.Button(
    description='Data',
    disabled=True,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Open Lab Data Sheet',
    icon='table' # (FontAwesome names without the `fa-` prefix)
)
output_widget = widgets.Output()
grades_output_widget = widgets.Output()

In [ ]:
# @title ipywidgets functions
from data_scrub.convert_utc_to_la import convert_utc_to_la
from IPython.display import display, Javascript, clear_output
from datetime import datetime

def open_doc(b):
    selected_lab_files = combined_lab_df[(combined_lab_df['user'] == user_dropdown.value) & (combined_lab_df['Course'] == course_dropdown.value) & (combined_lab_df['lab_number'] == lab_dropdown.value)]
    global selected_doc_url
    selected_doc_url = f"https://docs.google.com/document/d/{selected_lab_files['doc_id'].iloc[0]}/edit?tab=t.0"
    display(Javascript(f"window.open('{selected_doc_url}', '_blank');"))

def open_sheet(b):
    selected_lab_files = combined_lab_df[(combined_lab_df['user'] == user_dropdown.value) & (combined_lab_df['Course'] == course_dropdown.value) & (combined_lab_df['lab_number'] == lab_dropdown.value)]
    global selected_sheet_url
    selected_sheet_url = f"https://docs.google.com/spreadsheets/d/{selected_lab_files['sheet_id'].iloc[0]}/edit?gid=0#gid=0"
    display(Javascript(f"window.open('{selected_sheet_url}', '_blank');"))

def show_grades(show=False):
  with grades_output_widget:
    clear_output(wait=True)
    if show:
      selected_student_grades = grades_df[(grades_df['user'] == user_dropdown.value)]
      stripped_lab_number = str(int(lab_dropdown.value))

      lab_filter_string = f"Lab {stripped_lab_number}"
      lab_assignment_list = [col for col in grades_df.columns if lab_filter_string in col] +['Current Grade']

      global selected_lab_grades_df
      selected_lab_grades_df = selected_student_grades[lab_assignment_list]
      display(selected_lab_grades_df)

def on_used_data_change(change):
    with output_widget:
        show_grades(False)
        clear_output(wait=True)
        used_data_status = change.new
        # print(used_data_status)

        user_dropdown.options = ['Select User']
        user_dropdown.value = 'Select User'
        course_dropdown.options = ['Select Course']
        course_dropdown.value = 'Select Course'
        course_dropdown.disabled = True
        lab_dropdown.options = ['Select Lab']
        lab_dropdown.value = 'Select Lab'
        lab_dropdown.disabled = True
        question_dropdown.options = ['Select Question']
        question_dropdown.value = 'Select Question'
        question_dropdown.disabled = True
        doc_button.disabled = True
        sheet_button.disabled = True
        # print("Please select a User from the dropdown.")
        # return

        # Filter combined_df based on use_data_status
        if used_data.value:
          df_for_next_filter = combined_df[combined_df['data'] == used_data_status]
        else:
          df_for_next_filter = combined_df

        # Update course_dropdown options based on the user_filtered_df
        unique_user = df_for_next_filter['user'].dropna().unique().tolist()
        if len(unique_user) > 0:
            user_dropdown.options = ['Select User'] + sorted(unique_user)
            user_dropdown.value = 'Select User'
        else:
            user_dropdown.options = ['Select User']
            user_dropdown.value = 'Select User'

# This function will be called when the course dropdown changes
def on_user_change(change):
    with output_widget:
        show_grades(False)
        clear_output(wait=True)
        selected_user = change.new

         # Set Course options dependant on user selection
        if selected_user == "Select User":
            course_dropdown.options = ['Select Course']
            course_dropdown.value = 'Select Course'
            course_dropdown.disabled = True

            # print("Please select a User from the dropdown.")
            return
        else:
          # Apply used_data filter
          used_data_status = used_data.value
          if used_data.value:
            df_for_next_filter = combined_df[combined_df['data'] == used_data_status]
          else:
            df_for_next_filter = combined_df

          # Filter combined_df based on selected_course
          user_filtered_df = df_for_next_filter[df_for_next_filter['user'] == selected_user]

          # Update course_dropdown options based on the user_filtered_df
          unique_courses = user_filtered_df['Course'].dropna().unique().tolist()
          if len(unique_courses) > 0:
              course_dropdown.options = ['Select Course'] + sorted(unique_courses)
              course_dropdown.value = 'Select Course'
              course_dropdown.disabled = False

          else:
              course_dropdown.options = ['Select Course']
              course_dropdown.value = 'Select Course'
              course_dropdown.disabled = False


          # Reset Lab, question and user dropdowns (on_Course_change will populate them)
          # Setting course_dropdown.value already triggers on_course_change, which handles downstream updates
          lab_dropdown.options = ['Select Lab']
          lab_dropdown.value = 'Select Lab'
          lab_dropdown.disabled = True
          question_dropdown.options = ['Select Question']
          question_dropdown.value = 'Select Question'
          question_dropdown.disabled = True
          doc_button.disabled = True
          sheet_button.disabled = True

# This function will be called when the lab dropdown changes
def on_course_change(change):
    show_grades(False)
    with output_widget:
        clear_output(wait=True)
        selected_user = user_dropdown.value
        selected_course = change.new

        # Apply used_data filter
        used_data_status = used_data.value
        if used_data.value:
          df_for_next_filter = combined_df[combined_df['data'] == used_data_status]
        else:
          df_for_next_filter = combined_df

        # Apply User filter
        if selected_user != "Select User":
            df_for_next_filter = df_for_next_filter[df_for_next_filter['user'] == selected_user]

        # Apply course filter
        if selected_course != "Select Course":
            df_for_next_filter = df_for_next_filter[df_for_next_filter['Course'] == selected_course]

            # Update lab_dropdown options
            unique_labs = df_for_next_filter['lab_number'].dropna().unique().tolist()
            if len(unique_labs) > 0:
                lab_dropdown.options = ['Select Lab'] + sorted(unique_labs)
                lab_dropdown.value = 'Select Lab'
                lab_dropdown.disabled = False

            else:
                lab_dropdown.options = ['Select Lab']
                lab_dropdown.value = 'Select Lab'
                lab_dropdown.disabled = False
        else:
            lab_dropdown.options = ['Select Lab']
            lab_dropdown.value = 'Select Lab'
            lab_dropdown.disabled = True

        doc_button.disabled = True
        sheet_button.disabled = True
        # Reset question dropdown (on_lab_change will populate it)
        # Setting question_dropdown.value already triggers on_question_change, which handles downstream updates
        question_dropdown.options = ['Select Question']
        question_dropdown.value = 'Select Question'
        question_dropdown.disabled = True

# This function will be called when the question dropdown changes
def on_lab_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_user = user_dropdown.value
        selected_course = course_dropdown.value
        selected_lab = change.new

        # Apply used_data filter
        used_data_status = used_data.value
        if used_data.value:
          df_for_next_filter = combined_df[combined_df['data'] == used_data_status]
        else:
          df_for_next_filter = combined_df

        # Apply question filter
        if selected_user != "Select User":
            df_for_next_filter = df_for_next_filter[df_for_next_filter['user'] == selected_user]

        # Apply course filter
        if selected_course != "Select Course":
            df_for_next_filter = df_for_next_filter[df_for_next_filter['Course'] == selected_course]

        # Apply lab filter
        if selected_lab != "Select Lab":
            df_for_next_filter = df_for_next_filter[df_for_next_filter['lab_number'] == selected_lab]
            doc_button.disabled = False
            sheet_button.disabled = False

            selected_lab_files = combined_lab_df[(combined_lab_df['user'] == user_dropdown.value) & (combined_lab_df['Course'] == course_dropdown.value) & (combined_lab_df['lab_number'] == lab_dropdown.value)]
            global selected_doc_url
            global selected_sheet_url
            selected_doc_url = f"https://docs.google.com/document/d/{selected_lab_files['doc_id'].iloc[0]}/edit?tab=t.0"
            selected_sheet_url = f"https://docs.google.com/spreadsheets/d/{selected_lab_files['sheet_id'].iloc[0]}/edit?gid=0#gid=0"

            # Update question_dropdown options
            unique_questions = df_for_next_filter['question_number'].dropna().unique().tolist()
            if len(unique_questions) > 0:
                question_dropdown.options = ['Select Question'] + sorted(unique_questions)
                question_dropdown.value = 'Select Question'
                question_dropdown.disabled = False
            else:
                question_dropdown.options = ['Select Question']
                question_dropdown.value = 'Select Question'
                question_dropdown.disabled = False
            show_grades(True)

# This function will be called when the user dropdown changes
def on_question_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_user = user_dropdown.value
        selected_course = course_dropdown.value
        selected_lab = lab_dropdown.value
        selected_question = change.new


        # Don't filter the Question trans based on the Apply used_data filter
        # used_data_status = used_data.value
        # if used_data.value:
        #   df_to_display = combined_df[combined_df['data'] == used_data_status]
        # else:
        df_to_display = combined_df

        # Apply user filter
        if selected_user != "Select User":
            df_to_display = df_to_display[df_to_display['user'] == selected_user]
        else:
          print('Please select a user from the dropdown')
        # Apply course filter
        if selected_course != "Select Course":
            df_to_display = df_to_display[df_to_display['Course'] == selected_course]
        else:
          print('Please select a Course from the dropdown')

        # Apply lab filter
        if selected_lab != "Select Lab":
            df_to_display = df_to_display[df_to_display['lab_number'] == selected_lab]
        else:
          print('Please select a Lab from the dropdown')

        # Apply question filter
        if selected_question != "Select Question":
            df_to_display = df_to_display[df_to_display['question_number'] == selected_question]


            # Final display based on all filters
            user_str = f"User: {selected_user}"
            course_str = f"Course: {selected_course}"
            lab_str = f"Lab: {selected_lab}"
            question_str = f"Question: {selected_question}"

            print(f"Displaying data for {course_str}, {lab_str}, {question_str}, {user_str}.")
            if df_to_display.empty:
                print("No data matches the selected filters.")
            else:
                global trans_blocks
                trans_blocks = []
                # df_to_display = explode_json_messages(df_to_display)
                # display(df_to_display)
                for row in df_to_display.itertuples(index=False):
                  # process timestamp
                  timestamp_fixed = convert_utc_to_la(row.timestamp)
                  dt_obj = datetime.fromisoformat(str(timestamp_fixed))
                  natural_format = dt_obj.strftime("%b %d, %Y, %I:%M:%S %p")

                  print(f"\n{natural_format}\n{row.sender}: {row.text}\n")
                  trans_blocks.append(f"\n{natural_format}\n {row.sender}: {row.text}\n")
                  if row.sender == 'Bot':
                      row_dict = row._asdict()
                      # print(row_dict.keys())
                      if 'rating' in row_dict.keys() and str(row.rating) != 'nan':
                        print(f"Rating: {row.rating}")
                        trans_blocks.append(f"Rating: {row.rating}")
                      if 'data' in row_dict.keys() and row.data == True:
                        print(f"Spreadsheet data used: {row.data}")
                        trans_blocks.append(f"Spreadsheet data used: {row.data}")
                      print("------------------------\n")
                      trans_blocks.append("------------------------\n")
        else:
            print("Please Select a Question from the dropdown")
            
# --- Observers ---
# Link on_user_change to the user dropdown
user_dropdown.observe(on_user_change, names='value')
# Link on_course_change to the course dropdown
course_dropdown.observe(on_course_change, names='value')
# Link on_lab_change to the lab dropdown
lab_dropdown.observe(on_lab_change, names='value')
# Link on_question_change to the question dropdown
question_dropdown.observe(on_question_change, names='value')
# link doc_button to open_doc to open the doc url
doc_button.on_click(open_doc)
# link sheet_button to open_sheet to open the sheet url
sheet_button.on_click(open_sheet)
# Link on_used_data_change to the used_data checkbox
used_data.observe(on_used_data_change, names='value')

## Display Output

In [ ]:
# --- Display Widgets ---
left_box = widgets.VBox([user_dropdown, course_dropdown, lab_dropdown, question_dropdown])
right_box = widgets.VBox([doc_button,sheet_button])
display(widgets.VBox([used_data,widgets.HBox([left_box, right_box]),grades_output_widget]))

In [ ]:
# @title Chat Transcript
display(output_widget)

## Save selected output to file

### Drive output

In [ ]:
# @title export md to Drive {"form-width":"20%"}
drive_folder_link = "https://drive.google.com/drive/folders/1y57CGc5OI05-m48bg2Cw5jpdkbqeOIkN?usp=drive_link" # @param {"type":"string"}
drive_folder_id = extract_file_id(drive_folder_link)
export_md_file = False # @param {"type":"boolean"}
notes = "Student seems to have confused Q2 with image from Q1. LLM gave correct answer for Q1 (as the wording of their question implied). However, answer was incorrect for Q2. " # @param {"type":"string","placeholder":"Enter Observation notes here"}
if export_md_file:
  # Initialize selected_doc_url and selected_sheet_url if they are not defined
  # This handles the case where the cell is run before a specific lab is selected
  if 'selected_doc_url' not in globals():
      global selected_doc_url
      selected_doc_url = "N/A - Please select a specific Lab"
  if 'selected_sheet_url' not in globals():
      global selected_sheet_url
      selected_sheet_url = "N/A - Please select a specific Lab"

  header_blocks=[
        f"User: {user_dropdown.value}",
        f"Course: {course_dropdown.value}",
        f"Lab: Lab {lab_dropdown.value}",
        f"Question: Q{question_dropdown.value}",
        f"doc_url: {selected_doc_url}\n",
        f"sheet_url: {selected_sheet_url}\n"
        "\n---\n\n"
        ]
  footer_blocks = trans_blocks

  header_str = "\n".join(header_blocks)
  grades_str = selected_lab_grades_df.to_markdown(index=False)
  footer_str = "\n".join(footer_blocks)

  markdown_text = f"# Transcript Report\n\n{header_str}\n\n## Grades\n\n{grades_str}\n\n## Chat Transcript\n\n{footer_str}\n\n## Notes:\n\n{notes}"
  filename = f"{user_dropdown.value}-{course_dropdown.value}_Lab{lab_dropdown.value}_Q{question_dropdown.value}.md"

  # Upload to Google Drive if drive_folder_id is provided

  gdm = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)
  gdm.export_md(markdown_text, filename, drive_folder_id)
else:
  print("no file exported")